# 00 — Validação do ambiente

Este notebook confirma que o ambiente está pronto e que os cinco CSVs brutos podem ser lidos localmente. A pasta `data/` é tratada como somente leitura; nenhuma saída é gravada nela.

In [ ]:
from pathlib import Path
import hashlib

import matplotlib
import nbformat
import numpy as np
import pandas as pd
from IPython.display import display


def encontrar_raiz(inicio=None):
    caminho = Path(inicio or Path.cwd()).resolve()
    for candidato in (caminho, *caminho.parents):
        if (candidato / 'data').is_dir() and (candidato / 'ROADMAP.md').is_file():
            return candidato
    raise FileNotFoundError('Raiz do projeto não encontrada.')


RAIZ = encontrar_raiz()
DADOS = RAIZ / 'data'
SAIDAS = RAIZ / 'outputs'

ARQUIVOS_ESPERADOS = (
    'Details_Itapema.csv',
    'Hosts_ids_Itapema.csv',
    'Mesh_Ids_Data_Itapema.csv',
    'Price_AV_Itapema.csv',
    'VivaReal_Itapema.csv',
)

arquivos = [DADOS / nome for nome in ARQUIVOS_ESPERADOS]
ausentes = [arquivo.name for arquivo in arquivos if not arquivo.is_file()]
assert not ausentes, f'Arquivos ausentes em data/: {ausentes}'
assert SAIDAS.is_dir(), 'A pasta outputs/ não foi encontrada.'

print(f'Raiz do projeto: {RAIZ}')
print('Ambiente e estrutura básica validados.')

In [ ]:
def sha256(caminho, tamanho_bloco=1024 * 1024):
    resumo = hashlib.sha256()
    with caminho.open('rb') as arquivo:
        for bloco in iter(lambda: arquivo.read(tamanho_bloco), b''):
            resumo.update(bloco)
    return resumo.hexdigest()


inventario = pd.DataFrame(
    {
        'arquivo': [arquivo.name for arquivo in arquivos],
        'bytes': [arquivo.stat().st_size for arquivo in arquivos],
        'sha256': [sha256(arquivo) for arquivo in arquivos],
    }
)

# A leitura limitada valida formato e codificação sem iniciar a auditoria da Etapa 1.
amostras = {arquivo.name: pd.read_csv(arquivo, nrows=5) for arquivo in arquivos}
inventario['colunas'] = [len(amostras[nome].columns) for nome in inventario['arquivo']]
display(inventario)

In [ ]:
versoes = pd.Series(
    {
        'pandas': pd.__version__,
        'numpy': np.__version__,
        'matplotlib': matplotlib.__version__,
        'nbformat': nbformat.__version__,
    },
    name='versão',
)
display(versoes.to_frame())